# Corso di Big Data — Modulo Machine Learning per i Big Data
## Prova scritta del 4 giugno 2026

Si consideri il dataset `housing.csv`, allegato al presente compito, costituito da 20640 record che descrivono il valore medio di uno stabile in California sulla base delle seguenti caratteristiche:

* **longitude**
* **latitude**
* **housing_median_age**
* **total_rooms** (numero totale di camere nello stabile)
* **total_bedrooms** (numero totale di camere da letto nello stabile)
* **population** (numero di abitanti dello stabile)
* **households** (numero di famiglie nello stabile)
* **median_income**
* **median_house_value**
* **ocean_proximity**

---

### Esercizio 1 (Punti: __ / 8)
Eseguire uno split train/validation/test **80% / 5% / 15%**. 
Individuare eventuali dati mancanti, farne l'imputazione e procedere all'encoding più opportuno per le feature categoriche oltre all'eventuale scalatura di tutte le feature. 

Verificare la presenza di feature multicollineari e sostituirle con opportune combinazioni lineari. Infine eseguire la feature selection per individuare le caratteristiche più rilevanti del dataset attraverso una tecnica **embedded** che impieghi un regressore **Lasso** come modello. Infatti, sappiamo che la regressione lineare attribuisce coefficienti piccoli o nulli alle feature che influenzano poco la predizione.

---

### Esercizio 2 (Punti: __ / 6)
Implementare una regressione **Ridge** ed una con **Random Forest** per il dataset processato al punto 1. 
Confrontarne le prestazioni in termini di $Mio\_MSE$ (o MSE) e di $R^2$ sia sul training set sia sul test set.

---

### Esercizio 3 (Punti: __ / 10)
Implementare una piccola **rete neurale densa** con almeno due layer nascosti in **PyTorch**, utilizzando la semplice API `torchnn.py`, per eseguire la classificazione binaria. 

Si dovrà implementare anche il relativo `Dataset` per caricare i dati. La rete andrà addestrata per un massimo di 30 epoche. I soli layer nascosti dovranno avere un **dropout pari a 0.2**. Si utilizzi per l'addestramento l'ottimizzatore **Adam** con weight decay pari a $1 \times 10^{-4}$ e learning rate decrescente linearmente a partire da 0.01. Utilizzare l'**early stopping** con una pazienza sulla validation loss di 10 epoche e un incremento minimo di miglioramento pari a 0.001. Salvare il miglior modello rispetto al minimo MSE.

---

### Esercizio 4 (Punti: __ / 6)
Confrontare i risultati del regressore neurale sul test set con i modelli precedenti stampando i valori di MSE e $R^2$. 

Confrontare le performance dei tre regressori, stampando, per ciascuno di essi, il **boxplot** dei tre andamenti del MSE e di $R^2$ sui singoli campioni del test set.

---
**TOTALE: punti __ / 30**

### Esercizio 1 (Punti: __ / 8)
Eseguire uno split train/validation/test **80% / 5% / 15%**. 
Individuare eventuali dati mancanti, farne l'imputazione e procedere all'encoding più opportuno per le feature categoriche oltre all'eventuale scalatura di tutte le feature. 

Verificare la presenza di feature multicollineari e sostituirle con opportune combinazioni lineari. Infine eseguire la feature selection per individuare le caratteristiche più rilevanti del dataset attraverso una tecnica **embedded** che impieghi un regressore **Lasso** come modello. Infatti, sappiamo che la regressione lineare attribuisce coefficienti piccoli o nulli alle feature che influenzano poco la predizione

In [1]:
from sklearn.model_selection import train_test_split
import pandas as pd 

df = pd.read_csv('../../dbs/housing.csv')

print(df.shape)
df = df.dropna(subset=['median_house_value'])
print(df.shape)

X = df.drop( columns=['median_house_value'])
y = df.median_house_value

X_tr, X_te, y_tr, y_te = train_test_split(X,y, test_size=0.15,random_state=42)

val_size = (5/100) / (85/100)

X_tr, X_val , y_tr, y_val = train_test_split(X_tr,y_tr, test_size= val_size,random_state=42)

print(val_size)

(20640, 10)
(20640, 10)
0.05882352941176471


In [2]:
#print( X_tr.isna().sum())
#print( X_te.isna().sum())
#print(X_val.isna().sum())

#ho bissogno di fare imputazione

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder

imputer1 = SimpleImputer(strategy='median').set_output(transform='pandas')
imputer2 = SimpleImputer(strategy='constant').set_output(transform='pandas')

num_f = X_tr.select_dtypes( include=['number']).columns.tolist()
cat_f = [col for col in X_tr.columns if col not in num_f]


X_tr_num = imputer1.fit_transform( X_tr[num_f] )
X_te_num = imputer1.transform( X_te[num_f] )
X_val_num = imputer1.transform( X_val[num_f] )

X_tr_cat = imputer2.fit_transform( X_tr[cat_f] )
X_te_cat = imputer2.transform( X_te[cat_f] )
X_val_cat = imputer2.transform( X_val[cat_f] )

encoder = OrdinalEncoder().set_output(transform='pandas')

X_tr_cat = encoder.fit_transform( X_tr_cat )
X_te_cat = encoder.transform(  X_te_cat )
X_val_cat = encoder.transform( X_val_cat )

#rimetto i dataframe insieme

X_tr = pd.concat( [X_tr_num, X_tr_cat], axis=1)
X_te = pd.concat( [X_te_num, X_te_cat], axis=1)
X_val = pd.concat( [X_val_num, X_val_cat], axis=1)

print( X_tr.isna().sum())
print( X_te.isna().sum())
print(X_val.isna().sum())



longitude             0
latitude              0
housing_median_age    0
total_rooms           0
total_bedrooms        0
population            0
households            0
median_income         0
ocean_proximity       0
dtype: int64
longitude             0
latitude              0
housing_median_age    0
total_rooms           0
total_bedrooms        0
population            0
households            0
median_income         0
ocean_proximity       0
dtype: int64
longitude             0
latitude              0
housing_median_age    0
total_rooms           0
total_bedrooms        0
population            0
households            0
median_income         0
ocean_proximity       0
dtype: int64


In [3]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler().set_output(transform='pandas')

X_tr = scaler.fit_transform( X_tr )
X_te = scaler.transform( X_te )
X_val = scaler.transform( X_val )



In [4]:

corr_matrix = X_tr.corr().abs()

threshold = 0.7

#print(corr_matrix)

groups = [('longitude','latitude'),('population','total_bedrooms','total_rooms', 'households')]


for group in groups:
    print(group)
    for df in [X_tr,X_val,X_te]:
        df['_'.join(group)] = df[list(group)].mean(axis=1)
        df.drop(columns=list(group), inplace=True)

print(X_tr)



('longitude', 'latitude')
('population', 'total_bedrooms', 'total_rooms', 'households')
       housing_median_age  median_income  ocean_proximity  longitude_latitude  \
4990             1.539551      -1.169100        -0.818296           -0.070211   
397              1.698088      -0.002411         1.303739           -0.152608   
11153           -0.838494      -0.547109        -0.818296           -0.020491   
18732           -0.045813      -0.968031        -0.110951            0.437841   
18691           -0.679958      -0.182957        -0.818296           -0.238240   
...                   ...            ...              ...                 ...   
2883            -0.521422      -1.296411        -0.110951            0.090366   
12778            1.222478      -1.196852        -0.110951            0.242397   
5444             0.112724      -0.249171        -0.818296           -0.097812   
9260             0.271260      -1.277154        -0.110951            0.185943   
5704             0.74

In [5]:
from sklearn.linear_model import Lasso
from sklearn.feature_selection import SelectFromModel

model = Lasso(alpha=1.0)

sel = SelectFromModel(estimator=model, threshold='median')
print(X_tr.shape)

X_tr = sel.fit_transform( X_tr , y_tr)
X_te = sel.transform( X_te )
X_val = sel.transform( X_val )

print(X_tr.shape)

(16512, 5)
(16512, 3)


### Esercizio 2 (Punti: __ / 6)
Implementare una regressione **Ridge** ed una con **Random Forest** per il dataset processato al punto 1. 
Confrontarne le prestazioni in termini di $Mio\_MSE$ (o MSE) e di $R^2$ sia sul training set sia sul test set.

---

In [6]:
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error

regressor1 = Ridge(alpha=1.0)
regressor2 = RandomForestRegressor(n_estimators=50, criterion='squared_error', min_samples_split=100)

#1

model1 = regressor1.fit(X_tr, y_tr)

y_pred_1 = regressor1.predict( X_te)

score_r2_1 = r2_score( y_te, y_pred_1 )
mse_1 = mean_squared_error( y_te, y_pred_1 )

#2 

model2 = regressor2.fit(X_tr, y_tr)

y_pred_2 = regressor2.predict( X_te)

score_r2_2 = r2_score( y_te, y_pred_2 )
mse_2 = mean_squared_error( y_te, y_pred_2)

print(f'r2_1: {score_r2_1}')
print(f'r2_2: {score_r2_2}')
print(f'mse_1: {score_r2_1}')
print(f'mse_2: {score_r2_2}')

'''
r2_1: 0.5914939378073532
r2_2: 0.679487260452086
mse_1: 0.5914939378073532
mse_2: 0.679487260452086

il secondo modello ha statistiche migliori
'''

r2_1: 0.5793681533585692
r2_2: 0.6718699939360813
mse_1: 0.5793681533585692
mse_2: 0.6718699939360813


'\nr2_1: 0.5914939378073532\nr2_2: 0.679487260452086\nmse_1: 0.5914939378073532\nmse_2: 0.679487260452086\n\nil secondo modello ha statistiche migliori\n'

### Esercizio 3 (Punti: __ / 10)
Implementare una piccola **rete neurale densa** con almeno due layer nascosti in **PyTorch**, utilizzando la semplice API `torchnn.py`, per eseguire la classificazione binaria. 

Si dovrà implementare anche il relativo `Dataset` per caricare i dati. La rete andrà addestrata per un massimo di 30 epoche. I soli layer nascosti dovranno avere un **dropout pari a 0.2**. Si utilizzi per l'addestramento l'ottimizzatore **Adam** con weight decay pari a $1 \times 10^{-4}$ e learning rate decrescente linearmente a partire da 0.01. Utilizzare l'**early stopping** con una pazienza sulla validation loss di 10 epoche e un incremento minimo di miglioramento pari a 0.001. Salvare il miglior modello rispetto al minimo MSE.

---

In [7]:
import torchnn as utils
from torch.utils.data import Dataset
import numpy as np
import torch

class MyDataset(Dataset):
    
    def __init__(self,X,y):
        
        self.X = torch.tensor( np.array(X), dtype=torch.float32 )
        self.y = torch.tensor( np.array(y), dtype=torch.float32 )
        
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self,idx):
        return self.X[idx] , self.y[idx]
    
train_dl , val_dl , test_dl = utils.make_dataloaders(
    MyDataset(X_tr,y_tr),
    MyDataset(X_val,y_val),
    MyDataset(X_te, y_te)
)
        
        

Shape e tipo dei campioni: torch.Size([64, 3]), torch.float32
Shape e tipo delle etichette: torch.Size([64]) torch.float32


In [8]:
X_tr.shape[1]

3

In [9]:
from torch import nn


class Net(nn.Module):
    
    def __init__(self, depth,hidden_size):
        super().__init__()
        
        self.activation = nn.ReLU()
        layers = [ nn.Linear(X_tr.shape[1],hidden_size), self.activation ]
        for _ in range( depth - 2):
            layers += [nn.Dropout(0.2), nn.Linear(hidden_size,hidden_size), self.activation ]
            
        self.layers = nn.Sequential(*layers)
        self.output = nn.Linear(hidden_size,1)
        
    def forward(self,X):
        return self.output( self.layers(X) ) 
    
model = Net(4,64)
device = torch.device( 'cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), weight_decay=1e-4, lr=0.01)

early_stopping = utils.EarlyStopping(patience=10, min_delta=0.001)  
   
    

In [10]:
import copy

class SaveBestModel:
    
    def __init__(self):
        self.best_loss = float('inf')
        self.best_state_op = None
        self.best_state_model = None
        
    def __call__(self, model, opt, epoch, train_loss, val_loss, metrics):
        
        if val_loss < self.best_loss:
            self.best_loss = val_loss
            self.best_state_op = copy.deepcopy( opt.state_dict() ) 
            self.best_state_model = copy.deepcopy( model.state_dict() )
            self.epoch = epoch
            self.train_loss = train_loss
            self.metrics = metrics
            
    def save(self,path):
        torch.save({
            'best_loss' : self.best_loss,
            'best_state_op': self.best_state_op,
            'best_state_model': self.state_model,
            'epoch': self.epoch,
            'train_loss': self.train_loss,
            'metrics': self.metrics
        },path)

### Esercizio 3 (Punti: __ / 10)
Implementare una piccola **rete neurale densa** con almeno due layer nascosti in **PyTorch**, utilizzando la semplice API `torchnn.py`, per eseguire la classificazione binaria. 

Si dovrà implementare anche il relativo `Dataset` per caricare i dati. La rete andrà addestrata per un massimo di 30 epoche. I soli layer nascosti dovranno avere un **dropout pari a 0.2**. Si utilizzi per l'addestramento l'ottimizzatore **Adam** con weight decay pari a $1 \times 10^{-4}$ e learning rate decrescente linearmente a partire da 0.01. Utilizzare l'**early stopping** con una pazienza sulla validation loss di 10 epoche e un incremento minimo di miglioramento pari a 0.001. Salvare il miglior modello rispetto al minimo MSE.

---

 """ciclo di test/validazione su un'epoca

    Args:
        model (torch.nn.Module): classe del modello neurale
        dataloader (torch.utils.data.Dataloader): dataloader di test/validazione
        device (torch.device or int or str): device di esecuzione dell'addestramento
        loss_fn (optional): loss di test/validazione definita in ``torch.nn``. Defaults to config["test_loss"]
        metrics (list[callable(y_true, y_pred, **kwargs)], optional): lista delle metriche definite dall'utente che possono o meno avere un parametro average per la gestione delle metriche multiclasse. Defaults to config["metrics"]
        average (string|None, optional): parametro di calcolo delle metriche medie multiclasse. Segue la convenzione di sklearn: {‘micro’, ‘macro’, ‘samples’, ‘weighted’, ‘binary’} or None. Defaluts to config["metric_average"]

    Returns:
        tuple[float, float, dict[]]: loss di test/validazione, accuracy, dizionario dei valori delle metriche definite nella configurazione e calcolate sull'epoca
    """

In [11]:
from inspect import signature


def eval_loop(model, dataloader, device, loss_fn):
    model.eval()

    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, accuracy = 0.0, 0

    y_true = []
    y_pred = []

    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            accuracy += (pred.argmax(1) == y).type(torch.float).sum().item()
            y_true.extend(y.cpu().numpy())
            y_pred.extend(pred.argmax(1).cpu().numpy())
            

    test_loss /= num_batches
    accuracy /= size           

    epoch_metrics = dict()
    epoch_metrics['mse'] = mean_squared_error(y_true,y_pred)
    epoch_metrics['r2'] = r2_score(y_true,y_pred)

    return test_loss, epoch_metrics, y_true, y_pred


    """Ciclo di addestramento completo con test, early stopping e scheduler del learning rate

    Args:
        model (torch.nn.Module): classe del modello neurale
        optimizer (torch.optim.Optimizer): ottimizzatore
        device (torch.device or int or str): device di esecuzione dell'addestramento/test
        train_dataloader (torch.utils.data.Dataloader): dataloader di addestramento
        test_dataloader (torch.utils.data.Dataloader): dataloader di test
        epochs (int): numero di epoche. Defaults to config["epochs"]
        train_loss_fn (optional): loss di addestramento definita in ``torch.nn``. Defaults to config["train_loss"]
        test_loss_fn (optional): loss di test/validazione definita in ``torch.nn``. Defaults to config["test_loss"]
        early_stopping (EarlyStopping, optional): classe di early stopping se presente. Defaults to None.
        metrics (list[callable(y_true, y_pred, **kwargs)], optional): lista delle metriche definite dall'utente che possono o meno avere un parametro average per la gestione delle metriche multiclasse. Defaults to config["metrics"]
        average (string|None, optional): parametro di calcolo delle metriche medie multiclasse. Segue la convenzione di sklearn: {‘micro’, ‘macro’, ‘samples’, ‘weighted’, ‘binary’} or None. Defaluts to config["metric_average"]

    Returns:
        tuple[float, float, float, list[float], dict[list[float]]]: valori delle loss di train, test e validazione, lista dei valori di accuracy per epoca, dizionario con le liste dei valori delle metriche per epoca
    """

In [12]:



from tqdm import trange

scheduler = torch.optim.lr_scheduler.LinearLR(optimizer=optimizer,start_factor=0.1,end_factor=0.1,total_iters=30)

epochs = 30                                   
num_batches = len(train_dl.batch_sampler)
train_loss = []
metrics = {
    'mse' : [],
    'r2': []
}

save_best_model = SaveBestModel()


# Ciclo di addestramento con early stopping
for epoch in range(1,epochs+1):

    # Progress bar
    pbar = trange(num_batches)
    pbar.set_description(desc='Epoch {:4d}'.format(epoch))

    epoch_train_loss = utils.train_loop(model, train_dl, optimizer, device, pbar, loss_fn=criterion)
    train_loss.append(epoch_train_loss)
    
    val_loss, epoch_metrics, _, _ = eval_loop(model, val_dl, device, loss_fn=criterion)
   
    print(f"Epoch {epoch} - Train Loss: {epoch_train_loss:.4f} - Val Loss: {val_loss:.4f}")
    print(f"MSE: {epoch_metrics['mse']:.4f} - R2: {epoch_metrics['r2']:.4f}")
   
    metrics['mse'].append( epoch_metrics['mse'])
    metrics['r2'].append( epoch_metrics['r2'] )    
    
    #(self, model, opt, epoch, train_loss, val_loss, metrics):
    save_best_model(model, optimizer,epoch, epoch_train_loss, val_loss, metrics)

    # early stopping
    if early_stopping != None:
        early_stopping(val_loss)
        if early_stopping.early_stop:
            break

    # scheduler dell'iperparametro
    if scheduler is not None:
        scheduler.step()



Epoch    1:   0%|          | 0/258 [00:00<?, ?it/s]d:\Università\bigdata\workspace\big-data\Lib\site-packages\torch\nn\modules\loss.py:608: UserWarning: Using a target size (torch.Size([64])) that is different to the input size (torch.Size([64, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
Epoch    1: 100%|██████████| 258/258 [00:00<00:00, 305.30it/s]
d:\Università\bigdata\workspace\big-data\Lib\site-packages\torch\nn\modules\loss.py:608: UserWarning: Using a target size (torch.Size([8])) that is different to the input size (torch.Size([8, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Epoch 1 - Train Loss: 54823408139.9070 - Val Loss: 51885119367.5294
MSE: 58040716056.2500 - R2: -3.4033


Epoch    2:   0%|          | 0/258 [00:00<?, ?it/s]d:\Università\bigdata\workspace\big-data\Lib\site-packages\torch\nn\modules\loss.py:608: UserWarning: Using a target size (torch.Size([64])) that is different to the input size (torch.Size([64, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
Epoch    2: 100%|██████████| 258/258 [00:00<00:00, 362.40it/s]
d:\Università\bigdata\workspace\big-data\Lib\site-packages\torch\nn\modules\loss.py:608: UserWarning: Using a target size (torch.Size([8])) that is different to the input size (torch.Size([8, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Epoch 2 - Train Loss: 30127879044.9612 - Val Loss: 16308953208.4706
MSE: 58040716056.2500 - R2: -3.4033


Epoch    3:   0%|          | 0/258 [00:00<?, ?it/s]d:\Università\bigdata\workspace\big-data\Lib\site-packages\torch\nn\modules\loss.py:608: UserWarning: Using a target size (torch.Size([64])) that is different to the input size (torch.Size([64, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
Epoch    3: 100%|██████████| 258/258 [00:00<00:00, 375.11it/s]
d:\Università\bigdata\workspace\big-data\Lib\site-packages\torch\nn\modules\loss.py:608: UserWarning: Using a target size (torch.Size([8])) that is different to the input size (torch.Size([8, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


...

Epoch 21 - Train Loss: 13718539714.4806 - Val Loss: 13612435877.6471
MSE: 58040716056.2500 - R2: -3.4033


Epoch   22:   0%|          | 0/258 [00:00<?, ?it/s]d:\Università\bigdata\workspace\big-data\Lib\site-packages\torch\nn\modules\loss.py:608: UserWarning: Using a target size (torch.Size([64])) that is different to the input size (torch.Size([64, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
Epoch   22: 100%|██████████| 258/258 [00:00<00:00, 354.31it/s]
d:\Università\bigdata\workspace\big-data\Lib\site-packages\torch\nn\modules\loss.py:608: UserWarning: Using a target size (torch.Size([8])) that is different to the input size (torch.Size([8, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Epoch 22 - Train Loss: 13723443426.2326 - Val Loss: 13610730435.7647
MSE: 58040716056.2500 - R2: -3.4033


Epoch   23:   0%|          | 0/258 [00:00<?, ?it/s]d:\Università\bigdata\workspace\big-data\Lib\site-packages\torch\nn\modules\loss.py:608: UserWarning: Using a target size (torch.Size([64])) that is different to the input size (torch.Size([64, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
Epoch   23: 100%|██████████| 258/258 [00:00<00:00, 334.97it/s]
d:\Università\bigdata\workspace\big-data\Lib\site-packages\torch\nn\modules\loss.py:608: UserWarning: Using a target size (torch.Size([8])) that is different to the input size (torch.Size([8, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Epoch 23 - Train Loss: 13726830935.3178 - Val Loss: 12902677714.8235
MSE: 58040716056.2500 - R2: -3.4033


In [13]:
if save_best_model.best_state_model is not None:
    model.load_state_dict(save_best_model.best_state_model)
    optimizer.load_state_dict(save_best_model.best_state_op)
    
    
    
test_loss, test_metrics, y_true, y_pred = eval_loop(model, test_dl, device, loss_fn=criterion)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test MSE: {test_metrics['mse']:.4f} - Test R2: {test_metrics['r2']:.4f}")

Test Loss: 13132854794.4490
Test MSE: 55927917574.3333 - Test R2: -3.2677


d:\Università\bigdata\workspace\big-data\Lib\site-packages\torch\nn\modules\loss.py:608: UserWarning: Using a target size (torch.Size([64])) that is different to the input size (torch.Size([64, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
d:\Università\bigdata\workspace\big-data\Lib\site-packages\torch\nn\modules\loss.py:608: UserWarning: Using a target size (torch.Size([24])) that is different to the input size (torch.Size([24, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)



### Esercizio 4 (Punti: __ / 6)
Confrontare i risultati del regressore neurale sul test set con i modelli precedenti stampando i valori di MSE e $R^2$. 

Confrontare le performance dei tre regressori, stampando, per ciascuno di essi, il **boxplot** dei tre andamenti del MSE e di $R^2$ sui singoli campioni del test set.

In [14]:
import matplotlib.pyplot as plt
import numpy as np


# Assumiamo che tu abbia già i vettori delle predizioni sul Test Set:
# y_pred_ridge, y_pred_rf, y_pred_nn (e il target reale y_test)
# Convertiamo tutto in array numpy per sicurezza computazionale
y_true = np.array(y_te)

# --- 1. CALCOLO DEI VETTORI DI ERRORE QUADRATICO PUNTUALE (MSE) ---
mse_punti_ridge = (y_true - y_pred_1) ** 2
mse_punti_rf = (y_true - y_pred_2) ** 2


# --- 2. CALCOLO DEI VETTORI DI R2 PUNTUALE ---
varianza_totale = (y_true - np.mean(y_true)) ** 2
# Evitiamo divisioni per zero se un campione coincide perfettamente con la media
varianza_totale = np.where(varianza_totale == 0, 1e-9, varianza_totale)

r2_punti_ridge = 1 - (mse_punti_ridge / varianza_totale)
r2_punti_rf = 1 - (mse_punti_rf / varianza_totale)


# ==========================================
# FASE PLOT: 2 Figure distinte come richiesto
# ==========================================

# GRAFICO 1: Confronto Boxplot MSE
plt.figure(figsize=(10, 6))
dati_mse = [mse_punti_ridge, mse_punti_rf]
plt.boxplot(
    dati_mse,
    labels=["Ridge", "Random Forest"],
    showfliers=False,
)  # showfliers=False nasconde gli outlier estremi per rendere il grafico leggibile
plt.ylabel("Distanza Quadratica (MSE locale)")
plt.title("Confronto della distribuzione dell'Errore Quadratico sul Test Set")
plt.grid(True, alpha=0.3)
plt.show()

# GRAFICO 2: Confronto Boxplot R²
plt.figure(figsize=(10, 6))
dati_r2 = [r2_punti_ridge, r2_punti_rf]
plt.boxplot(
    dati_r2,
    labels=["Ridge", "Random Forest"],
    showfliers=False,
)
plt.ylabel("R² locale")
plt.title("Confronto della stabilità del Coefficiente R² sul Test Set")
plt.grid(True, alpha=0.3)
plt.show()

TypeError: boxplot() got an unexpected keyword argument 'labels'

<Figure size 1000x600 with 0 Axes>